# 07c — External POI / Land-Use Enrichment

Adds OpenStreetMap semantic context as an **independent** evidence axis for recurring non-HOME anchors in the support-qualified subset.

This notebook does **not** infer occupation, employment status, true WORK, or OFFICE.

OpenStreetMap data is used under the ODbL. Public outputs derived from this stage must attribute OpenStreetMap.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/tanh1c/geolife.git'
BRANCH = 'eda/07c-external-poi-landuse'
REPO_DIR = Path('/root/geolife')

if not (REPO_DIR / '.git').exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)], check=True)
else:
    subprocess.run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'checkout',BRANCH], check=True)
    subprocess.run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'], check=True)

print('repo:', REPO_DIR)
print('sha:', subprocess.run(['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'], check=True, capture_output=True, text=True).stdout.strip())


In [ ]:
from importlib.util import module_from_spec, spec_from_file_location
import pandas as pd

def load_module(name, path):
    spec = spec_from_file_location(name, path)
    assert spec is not None and spec.loader is not None
    module = module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

stage03a = load_module('stage03a_for_07c', REPO_DIR / 'analysis' / '03a_user_behavior_deep_dive.py')
stage07b = load_module('stage07b_for_07c', REPO_DIR / 'analysis' / '07b_factorized_work_profiles.py')
stage07c = load_module('stage07c', REPO_DIR / 'analysis' / '07c_external_poi_landuse_enrichment.py')
print('loaded Stage 03a / 07b / 07c helpers')


In [ ]:
VOLUME_ROOT = Path('/mnt/geolife-data')
CACHE_07C = VOLUME_ROOT / 'cache' / '07c_external_poi_landuse'
RAW_OSM_CACHE = CACHE_07C / 'overpass_raw'
CACHE_07C.mkdir(parents=True, exist_ok=True)
RAW_OSM_CACHE.mkdir(parents=True, exist_ok=True)

def find_required(filename, preferred=()):
    for path in preferred:
        if path.exists():
            return path
    matches = sorted(VOLUME_ROOT.glob(f'**/{filename}'))
    if matches:
        return matches[0]
    raise FileNotFoundError(f'Cannot find required artifact: {filename}')

def find_optional(filename, preferred=()):
    for path in preferred:
        if path.exists():
            return path
    matches = sorted(VOLUME_ROOT.glob(f'**/{filename}'))
    return matches[0] if matches else None

STAYS_PATH = find_required('stays_baseline_v1.pkl', [
    VOLUME_ROOT / 'cache' / '03a_user_behavior_deep_dive' / 'stays_baseline_v1.pkl',
    VOLUME_ROOT / 'artifacts' / '03a' / 'stays_baseline_v1.pkl',
])
FEATURES_PATH = find_required('user_behavior_features.csv', [
    VOLUME_ROOT / 'cache' / '03a_user_behavior_deep_dive' / 'user_behavior_features.csv',
])
HOME_PATH = find_required('home_consensus_private.pkl', [
    VOLUME_ROOT / 'cache' / '05b_home_consensus_adaptive_work' / 'home_consensus_private.pkl',
])
WORK_PATH = find_required('adaptive_work_patterns_42_private.pkl', [
    VOLUME_ROOT / 'cache' / '05b_home_consensus_adaptive_work' / 'adaptive_work_patterns_42_private.pkl',
])
ROUTINE_PATH = find_required('user_summary_private.pkl', [
    VOLUME_ROOT / 'cache' / '06_routine_habit_mining' / 'user_summary_private.pkl',
])
EDGE_PATH = find_required('edge_summary_private.pkl', [
    VOLUME_ROOT / 'cache' / '06_routine_habit_mining' / 'edge_summary_private.pkl',
])
INDEPENDENT_PATH = find_optional('candidate_comparison_private.pkl', [
    VOLUME_ROOT / 'cache' / '05c_stable_secondary_evidence' / 'candidate_comparison_private.pkl',
])
PROFILE_PATH = find_optional('factorized_work_profiles_private.pkl', [
    VOLUME_ROOT / 'cache' / '07b_factorized_work_profiles' / 'factorized_work_profiles_private.pkl',
])

print('private artifacts located; exact paths retained in runtime only')


In [ ]:
stays = pd.read_pickle(STAYS_PATH)
home_evidence = pd.read_pickle(HOME_PATH)
work_patterns = pd.read_pickle(WORK_PATH)
routine_users = pd.read_pickle(ROUTINE_PATH)
edge_summary = pd.read_pickle(EDGE_PATH)
independent = pd.read_pickle(INDEPENDENT_PATH) if INDEPENDENT_PATH else None

assert len(stays) == 5821, f'unexpected Stage-03a stay parity: {len(stays)}'

resolved = stage03a.resolve_stay_timezones(stays)
resolved = resolved.loc[resolved['timezone_id'].notna()].copy()
clustered, locations = stage03a.cluster_behavior_locations(resolved, 200.0)

if PROFILE_PATH is not None:
    profiles = pd.read_pickle(PROFILE_PATH)
else:
    behavior_features = pd.read_csv(FEATURES_PATH, dtype={'user_id': str})
    needed_flags = {'stable_shifted_candidate', 'mobile_work_like_candidate'}
    if not needed_flags.issubset(behavior_features.columns):
        try:
            import geolife  # noqa: F401
        except ImportError:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=REPO_DIR, check=True)
        behavior_features = stage03a.assign_behavioral_candidates(behavior_features)
    profiles = stage07b.run_audit(
        behavior_features,
        work_patterns,
        routine_users,
        home_evidence=home_evidence,
        independent_secondary_evidence=independent,
    ).profiles

candidate_anchors = stage07c.build_candidate_anchors(
    locations,
    home_evidence,
    profiles,
    work_patterns=work_patterns,
    edge_summary=edge_summary,
    min_stay_count=2,
)

summary = pd.DataFrame([{
    'candidate_users': candidate_anchors['user_id'].nunique(),
    'candidate_anchors': len(candidate_anchors),
    'stable_secondary_anchors': int(candidate_anchors['is_stable_secondary'].sum()),
    'repeated_route_anchors': int(candidate_anchors['appears_in_repeated_edge'].sum()),
    'route_region_geometry_users': int(candidate_anchors.loc[candidate_anchors['candidate_route_region_geometry'], 'user_id'].nunique()),
}])
display(summary)
assert not (candidate_anchors['location_id'] == candidate_anchors['home_location_id']).any()


## External query

The next cell sends only candidate anchor coordinates to a public Overpass endpoint to retrieve nearby OSM semantic objects. Queries are batched, sequential, and cached under the private Modal Volume. The notebook never prints coordinates or POI names.


In [ ]:
user_agent = os.environ.get(
    'OVERPASS_USER_AGENT',
    'geolife-stage07c-research/1.0 (github.com/tanh1c/geolife)',
)
primary_endpoint = os.environ.get('OVERPASS_ENDPOINT', stage07c.DEFAULT_ENDPOINT)
endpoint_candidates = [
    primary_endpoint,
    'https://overpass.kumi.systems/api/interpreter',
]
endpoint_candidates = list(dict.fromkeys(endpoint_candidates))

last_error = None
for endpoint in endpoint_candidates:
    try:
        osm_elements, request_log = stage07c.fetch_overpass_elements(
            candidate_anchors,
            cache_dir=RAW_OSM_CACHE,
            endpoint=endpoint,
            user_agent=user_agent,
            radius_m=250,
            batch_size=8,
            pause_s=1.25,
            timeout_s=120,
            retries=3,
        )
        print('Overpass batches:', len(request_log), '| semantic objects:', len(osm_elements))
        last_error = None
        break
    except Exception as exc:
        print('endpoint failed:', endpoint, '|', type(exc).__name__)
        last_error = exc

if last_error is not None:
    raise RuntimeError('All configured Overpass endpoints failed') from last_error


In [ ]:
audit = stage07c.run_audit_from_elements(
    candidate_anchors,
    osm_elements,
    query_radius_m=250,
    radii_m=(100, 250),
    request_log=request_log,
)

print('CONTEXT COVERAGE')
display(audit.coverage_summary)

print('CATEGORY SUMMARY — 100m')
display(audit.category_summary.sort_values(['cohort','present_share'], ascending=[True,False]))

print('STABLE SECONDARY vs SAME-USER PEERS — 100m')
display(audit.stable_secondary_summary)

print('CONTEXT SIGNATURES — 100m')
display(audit.signature_summary.head(30))

print('RADIUS SENSITIVITY')
display(audit.radius_sensitivity)


In [ ]:
# Exact coordinates, OSM IDs/tags, and user-level mappings remain private.
audit.candidate_anchors.to_pickle(CACHE_07C / 'candidate_anchors_private.pkl')
audit.osm_elements.to_pickle(CACHE_07C / 'osm_elements_private.pkl')
audit.anchor_element_links.to_pickle(CACHE_07C / 'anchor_element_links_private.pkl')
audit.anchor_context.to_pickle(CACHE_07C / 'anchor_context_private.pkl')
audit.stable_secondary_peer.to_pickle(CACHE_07C / 'stable_secondary_peer_private.pkl')
audit.request_log.to_pickle(CACHE_07C / 'overpass_request_log_private.pkl')

for name, frame in {
    'coverage_summary': audit.coverage_summary,
    'category_summary': audit.category_summary,
    'stable_secondary_summary': audit.stable_secondary_summary,
    'signature_summary': audit.signature_summary,
    'radius_sensitivity': audit.radius_sensitivity,
}.items():
    frame.to_csv(CACHE_07C / f'{name}.csv', index=False)

print('saved aggregate + private Stage-07c artifacts')


## Interpretation boundary

- `unknown` means OSM did not provide one of the queried semantic categories within the radius; it is not proof of absence.
- `work_compatible_context` is a broad diagnostic composite, not a WORK/OFFICE label.
- Nearby POI/land-use context does not identify occupation.
- Mobility axes from Stage 07b remain unchanged regardless of external context.

Data source: OpenStreetMap, available under the Open Database License (ODbL).
